# Titanic - Random Forest

Previsão de sobrevivência dos passageiros do Titanic com `RandomForestClassifier`
(competição [Titanic do Kaggle](https://www.kaggle.com/competitions/titanic)).
Baseado no tutorial [*Titanic Tutorial*](https://www.kaggle.com/code/alexisbcook/titanic-tutorial), de Alexis Cook (Kaggle), com features e validação adicionais.

**Etapas:** dados → exploração → feature engineering → modelo → validação → submissão.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## 1. Dados

Se `data/train.csv` não existir, o script `scripts/download_data.py` baixa os arquivos via Kaggle API
(veja o README para configurar as credenciais).

In [2]:
if not (DATA_DIR / "train.csv").exists():
    subprocess.run([sys.executable, "../scripts/download_data.py"], check=True)

train_data = pd.read_csv(DATA_DIR / "train.csv")
test_data = pd.read_csv(DATA_DIR / "test.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 2. Exploração

Taxa de sobrevivência por grupo.

In [4]:
women = train_data.loc[train_data.Sex == "female", "Survived"]
rate_women = women.mean()
print(f"% of women who survived: {rate_women:.2%}")

% of women who survived: 74.20%


In [5]:
men = train_data.loc[train_data.Sex == "male", "Survived"]
rate_men = men.mean()
print(f"% of men who survived: {rate_men:.2%}")

% of men who survived: 18.89%


In [6]:
age_group = train_data.loc[train_data.Age.between(20, 50), "Survived"]
rate_age_group = age_group.mean()
print(f"% of passengers aged 20-50 who survived: {rate_age_group:.2%}")

% of passengers aged 20-50 who survived: 38.89%


## 3. Feature engineering

- `FamilySize` = `SibSp` + `Parch` + 1 (o próprio passageiro).
- Valores faltantes de `Age` preenchidos com a mediana **do treino** (evita vazamento de dados).
- `Sex` convertido em colunas numéricas com `get_dummies`, alinhando treino e teste.

In [7]:
for df in (train_data, test_data):
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

y = train_data["Survived"]
features = ["Pclass", "Sex", "FamilySize", "Age"]

median_age = train_data["Age"].median()
X_raw = train_data[features].copy()
X_test_raw = test_data[features].copy()
X_raw["Age"] = X_raw["Age"].fillna(median_age)
X_test_raw["Age"] = X_test_raw["Age"].fillna(median_age)

X = pd.get_dummies(X_raw, dtype=int)
X_test = pd.get_dummies(X_test_raw, dtype=int)
X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)
X.head()

,Pclass,FamilySize,Age,Sex_female,Sex_male
0,3,2,22.0,0,1
1,1,2,38.0,1,0
2,3,1,26.0,1,0
3,1,2,35.0,1,0
4,3,1,35.0,0,1


## 4. Modelo

Random forest com 100 árvores e profundidade máxima 5 (limita overfitting).

In [8]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_STATE)
model.fit(X, y)
predictions = model.predict(X_test)

## 5. Validação

- **Validação cruzada** (5 folds) no conjunto de treino.
- **Hold-out**: 80% treino / 20% validação.

In [9]:
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"Mean accuracy (5-fold CV): {scores.mean():.4f} ({scores.mean():.2%})")

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
holdout_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_STATE)
holdout_model.fit(X_tr, y_tr)
val_accuracy = accuracy_score(y_val, holdout_model.predict(X_val))
print(f"Hold-out validation accuracy: {val_accuracy:.4f} ({val_accuracy:.2%})")

Mean accuracy (5-fold CV): 0.8283 (82.83%)
Hold-out validation accuracy: 0.8268 (82.68%)


## 6. Submissão

Gera `outputs/submission.csv` no formato exigido pelo Kaggle.

In [10]:
output = pd.DataFrame({
    "PassengerId": test_data["PassengerId"].to_numpy(),
    "Survived": predictions,
})
output.to_csv(OUTPUT_DIR / "submission.csv", index=False)
output.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## Conclusão

**Resultados**

| Validação | Acurácia |
|---|---|
| Validação cruzada (5 folds) | 82,83% |
| Hold-out (20%) | 82,68% |

As duas estimativas praticamente coincidem, o que indica que o modelo generaliza bem e não está sofrendo overfitting forte
(`max_depth=5` ajuda a limitar a complexidade).

**Observações da exploração**
- Sexo é o fator mais discriminante: 74,20% das mulheres sobreviveram contra 18,89% dos homens.
- A faixa etária de 20 a 50 anos teve 38,89% de sobrevivência.